# 00 — Data preparation
Read local recording-session CSVs directly from the configured shared Drive folder, then prepare all four forecast setups.
Only the folder's CSV files are read; filenames are unrestricted and subfolders are ignored. Source files are never modified or deleted.
Required columns: `elapsed_minutes`, `co2`, `temp`, `humidity`. Room is not used. Occupancy is not an input.
Run 00 → 01 → 02 → 05. All setups share eligible origin rows and the same whole-session 60/20/20 split.

`elapsed_minutes` must contain numeric elapsed minutes from the start of each recording. Clock-time features are not used.


In [ ]:
%pip -q install numpy pandas scikit-learn joblib google-api-python-client google-auth


In [ ]:
from pathlib import Path
import sys
import importlib
from urllib.request import urlopen

helper_candidates = [Path.cwd() / "kc_common.py", Path.cwd() / "KC_CO2_Modular_Notebooks" / "kc_common.py"]
helper_path = next((p for p in helper_candidates if p.is_file()), None)
if helper_path is None:
    HELPER_REF = "main"
    helper_url = f"https://raw.githubusercontent.com/Namelessr679/ESP32-Indoor-Air-Quality-1.0Monitor/{HELPER_REF}/KC_CO2_Modular_Notebooks/kc_common.py"
    try:
        with urlopen(helper_url, timeout=30) as response:
            helper_source = response.read()
    except Exception as exc:
        raise RuntimeError("kc_common.py is unavailable. Place the matching helper beside the notebook or select a published HELPER_REF, then rerun setup.") from exc
    compile(helper_source, "kc_common.py", "exec")
    if b"def prepare_experiment(" not in helper_source:
        raise RuntimeError("The downloaded helper is incompatible. Use kc_common.py from the same repository revision as these notebooks.")
    helper_path = Path.cwd() / "kc_common.py"
    helper_path.write_bytes(helper_source)
sys.path.insert(0, str(helper_path.parent))
import kc_common
if Path(kc_common.__file__).resolve() != helper_path.resolve():
    raise RuntimeError("A different kc_common module is already loaded. Restart the runtime and rerun setup.")
importlib.reload(kc_common)
from kc_common import experiment_run_dir, list_session_csvs, read_session_csv, clean_session, prepare_experiment


In [ ]:
from google.colab import drive
from IPython.display import display
import joblib
import pandas as pd

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
RUN_DIR = experiment_run_dir(PROJECT)


In [ ]:
# Read the shared data by folder ID; its display name and CSV names do not matter.
DATA_FOLDER_ID = "11FwkaIH_LhHWwaJsVcrCbBWwM1H32y-G"
DATA_FOLDER_URL = f"https://drive.google.com/drive/folders/{DATA_FOLDER_ID}"

from google.colab import auth
import google.auth
from googleapiclient.discovery import build

auth.authenticate_user()
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
data_drive = build("drive", "v3", credentials=credentials, cache_discovery=False)



csv_items = list_session_csvs(data_drive, DATA_FOLDER_ID)
if not csv_items:
    raise FileNotFoundError(f"No CSV files found in {DATA_FOLDER_URL}. Check your account's access.")
print("Reading shared data:", DATA_FOLDER_URL)
print("CSV files found:", len(csv_items))

raw_sessions = []
for item in csv_items:
    session = read_session_csv(data_drive, item)
    raw_sessions.append(clean_session(session, item["id"], item["name"]))

print("Loaded sessions:", len(raw_sessions))
print("Total dataset rows loaded:", sum(len(session) for session in raw_sessions))


## Prepare 5- and 10-minute direct and recursive forecasts


In [ ]:
prepared = prepare_experiment(raw_sessions, DATA_FOLDER_ID)
joblib.dump(prepared, RUN_DIR / "tree_session_prepared.joblib", compress=3)
joblib.dump(prepared["metadata"], RUN_DIR / "preparation_metadata.joblib")
summary = []
for key, config in prepared["configurations"].items():
    summary.append({"Setup": key, "Training rows": len(config["train_df"]),
                    "Validation rows": len(config["validation_df"]), "Test rows": len(config["test_df"])})
display(pd.DataFrame(summary))
print("Preparation:", prepared["metadata"]["run_id"])
